# Lab Exercise: Building Energy Atlas @Uni Graz with OSM + GHS-OBAT

**📝 Scenario.** The University of Graz Estates & Sustainability Office needs a *rapid, transparent, open-data proxy* to estimate and visualise building heating demand within **1 km** of campus. The output is used for **triage** (deciding which buildings deserve a detailed audit first), not for final retrofit design.

**🚚 What you will deliver**
1. A reproducible notebook that:
   - downloads building footprints from OpenStreetMap (OSM)
   - loads a subset of **GHS-OBAT** building attributes (epoch, height, use)
   - derives geometry/volume features with **explicit source tracking**
   - computes energy demand with **low/mid/high scenarios**
   - produces a Plotly map + a ranked table of candidates

2. A short advisory brief (<500 words) answering:
   - **top 3 buildings to prioritise** (based on your computed evidence and assumptions)
   - **technical + ethical risks** of relying on OSM + OBAT for policy
   - a **basic governance protocol** (“how to use this responsibly”)


<div class="alert alert-info">  

#### 🔗 Useful links:
- GHS data catalogue ([website](https://human-settlement.emergency.copernicus.eu/downloadWizard.php))
- GHS-OBAT paper ([pdf](https://www.sciencedirect.com/science/article/pii/S2352340925004780))
- TABULA Austria scientific report ([pdf](https://episcope.eu/fileadmin/tabula/public/docs/scientific/AT_TABULA_ScientificReport_AEA.pdf)) (typologies / construction periods)
- TABULA online ([website](https://webtool.building-typology.eu/#bm))

---

## 0. 🔧 Setup

---


In [ ]:
# Standard library
from pathlib import Path
import io
import json
import zipfile

# Download
import requests

# Data
import numpy as np
import pandas as pd

# Geo
import geopandas as gpd
from shapely.geometry import Point
from shapely import make_valid
import osmnx as ox

# Visualisation
import matplotlib.pyplot as plt
import plotly.express as px

# Folders
DATA_DIR = Path("./data")
OUTPUT_DIR = Path("./outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
CONFIG = {}

CONFIG["radius"] = 1000  # meters
CONFIG["crs_geographic"] = "EPSG:4326"
CONFIG["crs_projected"] = "EPSG:32633"

CONFIG["default_floor_height_m"] = 3.0
CONFIG["default_levels"] = 1

CONFIG["campus_query"] = "Universitätsplatz 3, 8010 Graz"
CONFIG["osm_cache_path"] = DATA_DIR / "osm_buildings.gpkg"
CONFIG["obat_csv_path"] = DATA_DIR / "GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0.csv"
CONFIG["obat_parquet_path"] = DATA_DIR / "obat.parquet"
CONFIG["obat_clip_path"] = DATA_DIR / "obat_clip.gpkg"
CONFIG["joined_path"] = OUTPUT_DIR / "joined_buildings.gpkg"
CONFIG

---

## 1. 🌍 Define the Area of Interest (AOI)

---

We work with a **1 km radius** around the campus point.

**Workflow:** 
1. get the coordinate of Uni Graz
2. build a point in WGS84
3. project to metres
4. buffer
5. keep the buffered polygon

In [ ]:
uni_gdf = ox.geocode_to_gdf(CONFIG["campus_query"])
uni_point = uni_gdf.geometry.iloc[0] # extract the geometry of the first element return by osmnx

campus_lat = float(uni_point.centroid.y)
campus_lon = float(uni_point.centroid.x)

In [ ]:
def make_aoi_polygon(lat: float, lon: float, radius_m: float, crs_proj: str) -> gpd.GeoDataFrame:
    """Create a circular AOI buffer around a point."""
    point_gdf = gpd.GeoDataFrame(
        {"name": ["campus"]},
        geometry=[Point(lon, lat)],
        crs="EPSG:4326",
    )
    point_proj = point_gdf.to_crs(crs_proj)
    aoi_proj = point_proj.buffer(radius_m).iloc[0]

    return gpd.GeoDataFrame(
        {"name": ["aoi"]},
        geometry=[aoi_proj],
        crs=crs_proj,
    )

In [ ]:
aoi = make_aoi_polygon(
    lat=campus_lat,
    lon=campus_lon,
    radius_m=CONFIG["radius"],
    crs_proj=CONFIG["crs_projected"],
)

In [ ]:
aoi_area_km2 = aoi.geometry.area.iloc[0] / 1_000_000
print(f"AOI area: {aoi_area_km2:.2f} km²")

ax = aoi.plot(facecolor="none", edgecolor="red", figsize=(6, 6))
plt.show()

---

## 2. 🏘️ Download building footprints from OpenStreetMap via OSMnx

---

In [ ]:
aoi_wgs84 = aoi.to_crs(CONFIG["crs_geographic"])
aoi_polygon_wgs84 = aoi_wgs84.geometry.iloc[0]

if CONFIG["osm_cache_path"].exists():
    osm = gpd.read_file(CONFIG["osm_cache_path"])
else:
    osm = ox.features_from_polygon(aoi_polygon_wgs84, tags={"building": True})
    osm.to_file(CONFIG["osm_cache_path"], driver="GPKG")

### 2.1 Cache and reload



In [ ]:
print(f"OSM rows: {len(osm)}")
print(osm.crs)
osm.head()

### 2.2 Minimal cleaning


In [ ]:
def clean_buildings(osm_gdf: gpd.GeoDataFrame, crs_proj: str) -> gpd.GeoDataFrame:
    """Keep valid polygon buildings and compute footprint area."""
    

    osm_gdf = osm_gdf[osm_gdf.geometry.notna() & ~osm_gdf.geometry.is_empty].copy()
    osm_gdf = osm_gdf[osm_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    osm_gdf["geometry"] = osm_gdf.geometry.apply(make_valid)
    osm_gdf = osm_gdf[osm_gdf.geometry.notna() & ~osm_gdf.geometry.is_empty].copy()

    osm_gdf = osm_gdf.to_crs(crs_proj)
    osm_gdf["footprint_area_m2"] = osm_gdf.geometry.area

    # Keep a stable ID for joins and later checks
    osm_gdf = osm_gdf.reset_index(drop=False).rename(columns={"index": "osm_id"})

    return osm_gdf

In [ ]:
osm_copy = osm.copy()

osm_clean = clean_buildings(osm_copy, CONFIG["crs_projected"])

assert (osm_clean["footprint_area_m2"] > 0).all(), "Some OSM buildings have non-positive area."
osm_clean[["osm_id", "footprint_area_m2"]].head()

### 2.3 Visualize your results 

In [ ]:
ax = aoi.plot(facecolor="none", edgecolor="red", figsize=(8, 8))
osm_clean.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.2)
plt.title("OSM buildings in the AOI")
plt.show()

---

## 3. 🗼 Download & Load GHS-OBAT attributes

---

### 3.1 Download dynamically OBAT data

In [ ]:
def download_zip(url: str, output_dir: str):
    """Download a zip file and extract it to the specified directory."""
    r = requests.get(url, timeout=30)
    r.raise_for_status() # raise an exception if the request failed
    with zipfile.ZipFile(io.BytesIO(r.content)) as zip_ref:
        zip_ref.extractall(output_dir)

In [ ]:
# Replace with the exact CSV zip URL you selected in the exercise
OBAT_ZIP_URL = "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_OBAT_GLOBE_R2024A/GHS_OBAT_CSV_GLOBE_R2024A/V1-0/GHS_OBAT_CSV_AUT_E2020_R2024A_V1_0.zip" # url might have changed

if not CONFIG["obat_csv_path"].exists():
    download_zip(OBAT_ZIP_URL, DATA_DIR)

# Load the extracted CSV
obat = pd.read_csv(CONFIG["obat_csv_path"])

# Save a parquet copy for faster reloads later
obat.to_parquet(CONFIG["obat_parquet_path"], index=False)

### 3.2 📊 Inspect schema and CRS

In [ ]:
print(obat.shape)
print(obat.dtypes)
obat.head()

### 3.3 📏 Clip OBAT to the AOI



In [ ]:
obat

In [ ]:
obat_gdf = gpd.GeoDataFrame(
    obat.copy(),
    geometry=gpd.points_from_xy(obat["lon"], obat["lat"]),
    crs="EPSG:4326",
)

# Reproject to the AOI CRS
obat_gdf = obat_gdf.to_crs(aoi.crs)

# First crop: AOI bounding box
minx, miny, maxx, maxy = aoi.total_bounds
obat_clip = obat_gdf.cx[minx:maxx, miny:maxy].copy()

obat_clip.to_file(CONFIG["obat_clip_path"], driver="GPKG")

print(f"OBAT before clipping: {len(obat_gdf)}")
print(f"OBAT after clipping: {len(obat_clip)}")

### 3.4 Visualize your results 

In [ ]:
ax = aoi.plot(facecolor="none", edgecolor="red", figsize=(8, 8))
obat_clip.plot(ax=ax, color="blue", markersize=2)
plt.title("OBAT points clipped to the AOI extent")
plt.show()

---

## 4. 🔗 Spatial Join OSM buildings to OBAT attributes

---

### 4.1 Matching options

**Option 1 — Centroid-in-polygon join**  
Match an OSM footprint to the OBAT feature whose centroid falls inside the OSM polygon.

**Option 2 — Nearest-neighbour join**  
Match by nearest centroid within a distance threshold.

For this lab exercise, we will implement **Option 1**. You can try **Option 2** as a bonus exercise.

In [ ]:
joined = gpd.sjoin(
    osm_clean,
    obat_clip,
    how="left",
    predicate="contains",
)

joined["has_obat"] = joined["index_right"].notna()
joined.head()


### 4.2 Match-quality report

In [ ]:
match_pct = joined["has_obat"].mean() * 100
print(f"Matched OSM buildings: {match_pct:.1f}%")

# Count how many OBAT points fall in each OSM building
obat_per_building = joined.groupby("osm_id")["index_right"].count()
print(f"Buildings with more than one OBAT point: {(obat_per_building > 1).sum()}")

In [ ]:
# Calculate the difference in are between OBAT & OSM
joined["area_ratio"] = joined["footprint_area_m2"] / joined["area_right"]
joined["area_ratio"].describe()

---

## 5. 🔥 Feature engineering: height, floors, GFA, heated volume (with source tracking)

---

For our energy estimation. We need:
- **Gross Floor Area (GFA, m²)**: proxy = footprint_area × number_of_floors
- **Gross Heated Volume (m³)**: proxy = footprint_area × height

##### Height estimation logic

Some data might be missing or incomplete. We need to handle these cases.
Most of our buildings will have a height from OBAT, but also a height from OSM. We need to prioritize which one to use and how to combine them. Because OBAT is an estimation based on 100m resolution data, we will prioritize the OSM height when available.

We will therefore compute: 
- IF OSM height is available, use it.
- IF OSM height is not available, use OBAT height.
- IF neither are available, and OSM has a `building:levels` attribute, use it. (we usually assume 3m per level)
- IF none are available, we will set the height to a default of 3m.

##### Source tracking

To keep the workflow transparent, we will also store **where each estimated value comes from** using `height_source`

This is useful later to:
- quantify uncertainty
- understand how much of the dataset depends on defaults
- explain limitations in the final brief

### 5.1 Height estimation

In [ ]:
def estimate_height(row: pd.Series, default_levels: int, default_floor_height_m: float) -> pd.Series:
    """Estimate building height and track its source."""
    osm_height = row.get("osm_height_m")
    obat_height = row.get("obat_height_m")
    osm_levels = row.get("osm_levels")

    if pd.notna(osm_height):
        height_m_est = float(osm_height)
        height_source = "osm_height"
    elif pd.notna(obat_height):
        height_m_est = float(obat_height)
        height_source = "obat_height"
    elif pd.notna(osm_levels):
        height_m_est = float(osm_levels) * default_floor_height_m
        height_source = "osm_levels"
    else:
        height_m_est = default_levels * default_floor_height_m
        height_source = "default"

    return pd.Series({
        "height_m_est": height_m_est,
        "height_source": height_source,
    })

In [ ]:
joined.rename(columns={'height_left': 'osm_height_m', 'building:levels': 'osm_levels', 'height_right': 'obat_height_m'}, inplace=True)

In [ ]:
joined[["height_m_est", "height_source"]] = joined.apply(
    estimate_height,
    axis=1,
    default_levels=CONFIG["default_levels"],
    default_floor_height_m=CONFIG["default_floor_height_m"],
)

joined["levels_est"] = np.maximum(
    1,
    np.round(joined["height_m_est"] / CONFIG["default_floor_height_m"]).astype(int),
)

joined["gfa_m2"] = joined["footprint_area_m2"] * joined["levels_est"]
joined["heated_volume_m3"] = joined["footprint_area_m2"] * joined["height_m_est"]

### 5.2 Sanity checks, source summary, and quick visualisation


In [ ]:
assert (joined["height_m_est"] >= 0).all(), "Some estimated heights are not positive."
assert (joined["gfa_m2"] >= 0).all(), "Some GFA values are not positive."
assert (joined["heated_volume_m3"] >= 0).all(), "Some heated volumes are not positive."

print("Height source summary:")
print(joined["height_source"].value_counts(dropna=False))

joined["height_source"].value_counts().plot(kind="bar", title="Height source")
plt.ylabel("Number of buildings")
plt.show()

joined["height_m_est"].plot(kind="hist", bins=30, title="Estimated building heights")
plt.xlabel("Height (m)")
plt.show()

---

## 6. 🏘️ Proxy energy model: simple TABULA archetypes

---

We do not know the real energy consumption of each building, so we have to build a **simple proxy model** for annual heating demand.

We will use TABULA typology system to get some coarse estimation, the rule of thumb we will apply is:  
1. **Older buildings** usually have higher heating demand.
2. **Compact residential blocks** usually perform better than small detached buildings

##### Simplified archetypes

We classify residential buildings into two broad groups:

- **`house_like`**  
  small footprint and low-rise buildings (SFH / TH)
- **`block_like`**  
  larger and/or taller residential buildings (MFH / AB)

This is a **proxy**, not a real building audit.


##### Step-by-step logic

1. assign a simple archetype from building size
2. map `(epoch, archetype)` to a specific demand in `kWh/m²a`
3. compute annual heating demand:

   `annual_heat_kwh = gfa_m2 × specific_demand_kwh_m2a`
4. convert to MWh (/ 1000)
5. rank buildings by annual heat demand

##### Why rank by annual heat demand?

We will rank by **absolute annual heat demand** (`annual_heat_mwh`).

This is a simple and defensible choice because:
- the client wants to identify buildings with the **largest heating burden**
- it is easy to interpret
- it avoids adding another complex scoring model

The drawback is that this favours **large buildings**.


In [ ]:
residential = joined[joined["use"] == 1].copy()
residential["obat_epoch"] = pd.to_numeric(residential["epoch"], errors="coerce")

### 6.1 Building archetypes

Create a **simple residential archetype** for each building.

A building is:
- `house_like` if it has **2 floors or fewer** and **small footprint** (e.g., < 300 m²)
- `block_like` otherwise

This is only a proxy to support the energy estimation.

In [ ]:
def classify_simple_archetype(row: pd.Series) -> str:
    """Classify a residential building into a simple archetype."""
    if row["levels_est"] <= 2 and row["footprint_area_m2"] < 250:
        return "house_like"
    return "block_like"

residential["simple_archetype"] = residential.apply(classify_simple_archetype, axis=1)

print(residential["simple_archetype"].value_counts())

### 6.2 Lookup and energy estimation



In [ ]:
demand_lookup = {
    "house_like": {
        1: 150,
        2: 110,
        3: 80,
        4: 60,
        5: 40,
    },
    "block_like": {
        1: 120,
        2: 95,
        3: 70,
        4: 50,
        5: 35,
    },
}


In [ ]:
def get_specific_demand(row: pd.Series) -> float:
    """Return the specific heating demand from archetype and epoch."""
    archetype = row["simple_archetype"]
    epoch = int(row["obat_epoch"]) if pd.notna(row["obat_epoch"]) else None
    return demand_lookup.get(archetype, {}).get(epoch, np.nan)

residential["specific_demand_kwh_m2a"] = residential.apply(get_specific_demand, axis=1)
residential["annual_heat_kwh"] = residential["gfa_m2"] * residential["specific_demand_kwh_m2a"]
residential["annual_heat_mwh"] = residential["annual_heat_kwh"] / 1000


In [ ]:
assert residential["specific_demand_kwh_m2a"].notna().all(), "Some residential buildings have no demand value."
assert (residential["annual_heat_mwh"] > 0).all(), "Some annual heat values are not positive."

### 6.3 Rank retrofit candidates

We now rank buildings to identify the **top retrofit candidates**.

### Ranking rule

- **annual heating demand** (`annual_heat_mwh`)


In [ ]:
ranking = residential.sort_values("annual_heat_mwh", ascending=False).copy()

top10 = ranking[
    ["simple_archetype", "obat_epoch", "gfa_m2", "specific_demand_kwh_m2a", "annual_heat_mwh", "geometry"]
].head(10)

top10

In [ ]:
top10.plot(
    kind="bar",
    x="simple_archetype",
    y="annual_heat_mwh",
    legend=False,
    title="Top 10 retrofit candidates by annual heat demand",
)
plt.ylabel("Annual heat demand (MWh)")
plt.show()

In [ ]:
ax = aoi.plot(facecolor="none", edgecolor="red", figsize=(8, 8))
osm_clean.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.2)
top10.plot(ax=ax, color="yellow", edgecolor="orange", linewidth=0.5)
plt.title("Top 10 Buildings by Energy Efficiency")
plt.show()

---

## 7. 📊 Interactive visualization with Plotly

---

Until now, we have used static plots. These are useful for quick checks, but they do not let the user explore the data in detail.

In this section, we use **Plotly**, a Python library for creating **interactive visualizations**.

##### What is Plotly?

[Plotly](https://plotly.com/python/) is a visualization library that allows you to create charts and maps that users can interact with directly.

For example, interactivity can allow the user to:
- move around the figure
- zoom in and out
- hover over an element to see more information
- inspect individual buildings more easily

This is especially useful in spatial data science, where we often want to explore data at different scales and compare individual features.

##### Why use Plotly here?

Our goal is not only to compute building energy demand, but also to **communicate** the results clearly.

A static map shows the general pattern.  
An interactive map allows the user to:
- inspect a single building
- compare neighbouring buildings
- explore the relationship between geometry and estimated energy demand

### 7.1 Recreate one earlier chart with Plotly

Before building the map, let's recreate one of your earlier results with Plotly.

For example:
- the top 10 retrofit candidates,
- the distribution of estimated heights
- the count of buildings by height source

This helps you compare:
- **static plotting** (quick and simple)
- **interactive plotting** (better for exploration)

In [ ]:
# create a new ordered index for the top10 dataframe
top10 = top10.reset_index(drop=True)
top10["rank"] = top10.index + 1

In [ ]:
fig = px.bar(
    top10,
    x="rank",
    y="annual_heat_mwh",
    color="simple_archetype",
    hover_data=["gfa_m2", "specific_demand_kwh_m2a", "simple_archetype"],
    title="Top 10 retrofit candidates",
    width=900, height=500
)

fig.show()

### 7.2 Create an interactive choropleth map

Now create an interactive map where building geometries are coloured by your chosen energy metric.

Recommended metric:
- `annual_heat_mwh`

A **[choropleth map](https://datavizcatalogue.com/methods/choropleth.html)** is a type of thematic map where features are coloured according to a value.

Here, each building polygon will be coloured by its estimated annual heating demand.

In [ ]:
ranking.geometry

In [ ]:
ranking = ranking.to_crs("EPSG:4326").copy()
ranking = ranking.reset_index(drop=True)

fig_map = px.choropleth_map(
    ranking,
    geojson=ranking.geometry,
    locations=ranking.index,
    color="annual_heat_mwh",
    hover_data={
        "footprint_area_m2": True,
        "levels_est": True,
        "height_m_est": True,
        "obat_epoch": True,
        "annual_heat_mwh": True,
    },
    center={"lat": campus_lat, "lon": campus_lon},
    zoom=14,
    opacity=0.7,
    title="Estimated annual heating demand by building",
    height=900
)

fig_map.show()

### 7.3 🥊 (optional) 3D exploration

Plotly does not easily extrude polygons in a free map view without extra complexity.
Instead, we can produce an interactive **3D scatter** of building centroids:
- x, y = projected coordinates (metres), you can use the centroid of the geometry
- z = `height_m_est`
- color = `building_usage`
This is not a map, but it helps explore the vertical structure! 

You can find some information about a 3D scatter plot with plotly, [here](https://plotly.com/python/3d-scatter-plots/).

In [ ]:
centroids = ranking.copy()
centroids["centroid"] = centroids.geometry.centroid
centroids["x"] = centroids["centroid"].x
centroids["y"] = centroids["centroid"].y

fig_3d = px.scatter_3d(
    centroids,
    x="x",
    y="y",
    z="height_m_est",
    color="annual_heat_mwh",
    hover_data=["gfa_m2", "simple_archetype", "obat_epoch"],
    title="3D exploration of estimated building height",
)

fig_3d.show()